# 本場綱要校正（M1）：開發集檢查

候選功能：房主在聚會前貼上當堂綱要，字幕校正多兩件事——
1. 綱要裏用到的內建專名（書卷、教會專名、聖經人名）以及經節引用的書卷，**本場**也接受讀音相近（聲調、台灣混音）的辨識結果；
2. 綱要中重複出現、一般詞典沒有的詞（人名、地名、專門用語）加入同音校正。

**不需要 GPU，也不用 Cloudflare 額度**：沿用雲端硬碟上生命讀經開發集 16 篇的辨識結果，只重新計分。
每篇的「模擬綱要」＝篇題＋小標＋全篇經節（不含內文），存在結果資料夾的 `NN.outline.md`。

另外拿每篇的**正確原文**本身跑校正：綱要多改的地方都是誤改。

跑完回覆「直接讀」，我讀結果資料夾的 `summary.md`。MEET 封存測試沒有真實綱要，這本不跑 MEET。

In [ ]:
# 第 1 格：設定（目前只有開發集）
STAGE = 'dev'

In [ ]:
# 第 2 格：連接 Google Drive、找到資料
import os, subprocess, sys
from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive')
ARCHIVE = ROOT/'twgbr-archive'
if not (ARCHIVE/'life-study').is_dir():
    found = [Path(d) for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and 'life-study' in dirs and 'recovery-version' in dirs]
    if not found:
        raise RuntimeError('找不到 twgbr-archive（含 life-study 與 recovery-version）')
    ARCHIVE = found[0]

def find(name):
    for path in (ROOT/name, ARCHIVE/name):
        if path.is_dir():
            return path
    found = [Path(d)/name for d, dirs, _ in os.walk(ROOT) if len(Path(d).relative_to(ROOT).parts) < 3 and name in dirs]
    if not found:
        raise RuntimeError(f'找不到 {name} 資料夾')
    return found[0]

def latest_run(folder, pattern):
    runs = sorted(p for p in (folder/'eval-runs').glob('*') if p.is_dir() and any(p.glob(pattern))) if (folder/'eval-runs').is_dir() else []
    if not runs:
        raise RuntimeError(f'{folder}/eval-runs 沒有 {pattern} 的辨識結果')
    return runs[-1]

DATA = find('life-study-dev-v1')
EARLIER = latest_run(DATA, '*.whisper_live_vad_10s.json')
CONDITIONS, EXTRA = 'vad10', ['--development', '--loose-reference', '--outline']
print('資料：', DATA, '\n沿用辨識結果：', EARLIER)


In [ ]:
# 第 3 格：取得固定版本的程式與套件（約 1 分鐘）
COMMIT = 'b291e7f5dd55dc11d2c243550759f6e199d81522'
REPO = Path('/content/Meeting-Live-Captions')
if not REPO.exists():
    subprocess.run(['git', 'clone', '-q', 'https://github.com/timo893131/Meeting-Live-Captions.git', str(REPO)], check=True)
subprocess.run(['git', 'fetch', '-q', 'origin', COMMIT], cwd=REPO, check=True)
subprocess.run(['git', 'checkout', '-q', '--detach', COMMIT], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rapidfuzz==3.14.1', 'opencc-python-reimplemented==0.1.7',
                'nodejs-wheel-binaries==22.20.0'], check=True)
import nodejs_wheel
NODE = str(Path(nodejs_wheel.__file__).parent/'bin'/'node')
os.environ['NODE_BINARY'] = NODE
print('程式版本', COMMIT[:12])

In [ ]:
# 第 4 格：重新計分（不重新辨識；約 2–5 分鐘）
RUN = DATA/'eval-runs'/(datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S') + '-outline')
command = [sys.executable, '-u', str(REPO/'scripts/evaluate_meet.py'), '--meet', str(DATA), '--output', str(RUN),
           '--reuse', str(EARLIER), '--conditions', CONDITIONS] + EXTRA
process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=REPO)
for line in process.stdout:
    print(line, end='', flush=True)
if process.wait():
    raise RuntimeError('計分失敗，請看上面的錯誤訊息')
print('\n結果：', RUN/'summary.md', '\n請看 phonetic（網站現在）對 phonetic_outline（加上綱要）那兩行，以及「正確原文的誤改」。')